# 15 — Descriptive statistics

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                    if (p / "analysis" / "lib").exists())
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from scipy import stats

CLIN_PATH = PROJECT_ROOT / "data" / "tabular" / "clinical_data.csv"
OUT_DIR   = PROJECT_ROOT / "outputs" / "descriptives"
OUT_DIR.mkdir(parents=True, exist_ok=True)
assert CLIN_PATH.exists(), f"missing {CLIN_PATH} — see data/README.md"

clinical = pd.read_csv(CLIN_PATH, index_col=0)
clinical = clinical[clinical["cognition-PSCI_present"].notnull()]
print(f"{len(clinical)} subjects, {clinical.shape[1]} columns")

In [ ]:
# (column, kind, label); kind drives the cell renderer
VAR_SPEC = [
    ("demographics-age",                 "continuous",  "Age (years)"),
    ("demographics-sex",                 "categorical", "Sex"),
    ("demographics-ethnicity",           "categorical", "Ethnicity"),
    ("demographics-education_STROKOG",   "categorical", "Education (STROKOG)"),
    ("stroke-handedness",                "categorical", "Handedness"),
    ("stroke-NIHSS",                     "continuous",  "NIHSS"),
    ("stroke-prior_stroke",              "binary",      "Prior stroke"),
    ("stroke-prior_TIA",                 "binary",      "Prior TIA"),
    ("stroke-interval_onset_NPO_days",   "continuous",  "Onset → NPO (days)"),
    ("stroke-interval_onset_MRI_days",   "continuous",  "Onset → MRI (days)"),
    ("stroke-total_infarct_volume_ml",   "continuous",  "Total infarct volume (ml)"),
    ("vascularrisk-obesity",             "binary",      "Obesity"),
    ("vascularrisk-smoking_present",     "binary",      "Smoking (current)"),
    ("vascularrisk-smoking_past",        "binary",      "Smoking (past)"),
    ("vascularrisk-hypertension",        "binary",      "Hypertension"),
    ("vascularrisk-hyperlipidemia",      "binary",      "Hyperlipidemia"),
    ("vascularrisk-diabetes_mellitus",   "binary",      "Diabetes mellitus"),
    ("cognition-PSCI_present",                "binary", "PSCI present"),
    ("cognition-aef_domain_score",            "binary", "Impaired: attention / EF"),
    ("cognition-ps_domain_score",             "binary", "Impaired: processing speed"),
    ("cognition-language_domain_score",       "binary", "Impaired: language"),
    ("cognition-verbalmemory_domain_score",   "binary", "Impaired: verbal memory"),
    ("cognition-vsfunctions_domain_score",    "binary", "Impaired: visuospatial fn."),
    ("cognition-vsmemory_domain_score",       "binary", "Impaired: visuospatial mem."),
    ("cognition-aef_domain_zscore",           "continuous", "z-score: attention / EF"),
    ("cognition-ps_domain_zscore",            "continuous", "z-score: processing speed"),
    ("cognition-language_domain_zscore",      "continuous", "z-score: language"),
    ("cognition-verbalmemory_domain_zscore",  "continuous", "z-score: verbal memory"),
]
missing_cols = [v[0] for v in VAR_SPEC if v[0] not in clinical.columns]
print(f"missing in clinical: {missing_cols or 'none'}")

In [ ]:
def fmt_continuous(s):
    s = s.dropna()
    if len(s) == 0:
        return "—"
    median = float(s.median())
    q1, q3 = float(s.quantile(0.25)), float(s.quantile(0.75))
    mean = float(s.mean())
    sd = float(s.std(ddof=1)) if len(s) > 1 else float("nan")
    return f"{median:.2f} [{q1:.2f}–{q3:.2f}] (mean {mean:.2f} ± {sd:.2f})"


def fmt_binary(s):
    s = s.dropna()
    if len(s) == 0:
        return "—"
    n_pos = int((s == 1).sum())
    return f"{n_pos} ({100.0 * n_pos / len(s):.1f}%)"


def fmt_categorical_per_level(s, level):
    s = s.dropna()
    if len(s) == 0:
        return "—"
    n = int((s == level).sum())
    return f"{n} ({100.0 * n / len(s):.1f}%)"


def n_missing(s):
    n_na = int(s.isna().sum())
    if n_na == 0:
        return ""
    return f"{n_na} ({100.0 * n_na / len(s):.1f}%)"


def kruskal_p(s, cohort):
    df = pd.DataFrame({"v": s, "g": cohort}).dropna()
    groups = [g["v"].values for _, g in df.groupby("g") if len(g) > 1]
    if len(groups) < 2 or sum(len(g) for g in groups) < 5:
        return np.nan
    try:
        return float(stats.kruskal(*groups).pvalue)
    except Exception:
        return np.nan


def chi2_p(s, cohort):
    df = pd.DataFrame({"v": s.astype("category"), "g": cohort}).dropna()
    if df["v"].nunique() < 2 or df["g"].nunique() < 2:
        return np.nan
    table = pd.crosstab(df["g"], df["v"])
    if table.size == 0:
        return np.nan
    try:
        chi2, p, dof, expected = stats.chi2_contingency(table)
        if expected.min() < 5 and table.shape == (2, 2):
            _, p = stats.fisher_exact(table.values)
        return float(p)
    except Exception:
        return np.nan


def fmt_p(p):
    if p is None or not (p == p):
        return "n/a"
    return "<0.001" if p < 0.001 else f"{p:.3g}"

In [ ]:
cohort = clinical["demographics-cohort"]
cohort_levels = sorted(cohort.unique())
rows = []


def cohort_subset(col):
    return {c: clinical.loc[cohort == c, col] for c in cohort_levels}


hdr = {"Variable": "N", "Overall": f"{len(clinical)}"}
for c in cohort_levels:
    hdr[c] = f"{int((cohort == c).sum())}"
hdr["p"] = ""
rows.append(hdr)

for col, kind, label in VAR_SPEC:
    s_all = clinical[col]
    sub = cohort_subset(col)
    if kind == "continuous":
        rows.append({"Variable": label, "Overall": fmt_continuous(s_all),
                     **{c: fmt_continuous(sub[c]) for c in cohort_levels},
                     "p": fmt_p(kruskal_p(s_all, cohort))})
    elif kind == "binary":
        rows.append({"Variable": label, "Overall": fmt_binary(s_all),
                     **{c: fmt_binary(sub[c]) for c in cohort_levels},
                     "p": fmt_p(chi2_p(s_all, cohort))})
    elif kind == "categorical":
        rows.append({"Variable": label, "Overall": "",
                     **{c: "" for c in cohort_levels},
                     "p": fmt_p(chi2_p(s_all, cohort))})
        for lvl in sorted(s_all.dropna().unique()):
            level_str = (str(int(lvl)) if isinstance(lvl, float)
                         and float(lvl).is_integer() else str(lvl))
            rows.append({"Variable": f"  · {level_str}",
                         "Overall": fmt_categorical_per_level(s_all, lvl),
                         **{c: fmt_categorical_per_level(sub[c], lvl)
                            for c in cohort_levels},
                         "p": ""})
    miss = n_missing(s_all)
    if miss:
        rows.append({"Variable": "  · Missing", "Overall": miss,
                     **{c: n_missing(sub[c]) for c in cohort_levels}, "p": ""})

table = pd.DataFrame(rows, columns=["Variable", "Overall"] + cohort_levels + ["p"])
print(f"{len(table)} rows × {table.shape[1]} columns")

In [ ]:
table.to_csv(OUT_DIR / "clinical_table1.csv", index=False)
styled = (table.style
          .set_caption("Characteristics of the study population by cohort")
          .hide(axis="index")
          .set_table_styles([
              {"selector": "caption",
               "props": "font-weight: bold; font-size: 1.1em; padding: 6px;"},
              {"selector": "th",
               "props": "text-align: left; background: #f0f0f0; padding: 4px 8px;"},
              {"selector": "td",
               "props": "font-family: monospace; padding: 3px 8px;"},
              {"selector": "tr:nth-child(even)", "props": "background: #fafafa;"},
          ]))
(OUT_DIR / "clinical_table1.html").write_text(styled.to_html())
try:
    (OUT_DIR / "clinical_table1.md").write_text(table.to_markdown(index=False))
except ImportError:
    cols = list(table.columns)
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, row in table.iterrows():
        lines.append("| " + " | ".join(str(v) if v == v else "" for v in row.values) + " |")
    (OUT_DIR / "clinical_table1.md").write_text("\n".join(lines) + "\n")
print(f"saved clinical_table1.{{csv,md}} to {OUT_DIR}")
table

In [ ]:
# Stratum sizes for the age- and sex-stratified analyses.
# Age tertile cuts as in notebook 10 (acute infarct pool); counts in the
# analysed sample (any-domain PSCI available).
from analysis.lib.data import restricted_pool

LACUNA_DIR = PROJECT_ROOT / "data" / "lacuna" / "METAVCI_PSCI_LACUNA" / "derivatives"
clin_all = pd.read_csv(CLIN_PATH, index_col=0)
acute_pool = restricted_pool(LACUNA_DIR, clin_all, ["acuteinfarct"])
age_pool = pd.to_numeric(clin_all.loc[acute_pool, "demographics-age"], errors="coerce")
a1, a2 = age_pool.quantile([1 / 3, 2 / 3])

analysed = clinical.index.intersection(acute_pool)
age = pd.to_numeric(clinical.loc[analysed, "demographics-age"], errors="coerce")
sex = pd.to_numeric(clinical.loc[analysed, "demographics-sex"], errors="coerce")
sizes = pd.DataFrame({
    "stratum": ["age_young", "age_middle", "age_old", "sex_0", "sex_1"],
    "definition": [f"age < {a1:g}", f"{a1:g} <= age < {a2:g}", f"age >= {a2:g}",
                   "sex == 0", "sex == 1"],
    "n": [int((age < a1).sum()), int(((age >= a1) & (age < a2)).sum()),
          int((age >= a2).sum()), int((sex == 0).sum()), int((sex == 1).sum())],
})
sizes.to_csv(OUT_DIR / "strata_sizes.csv", index=False)
print(sizes.to_string(index=False))
